# Module 1, Video 4: Vector Store Comparison
## pgvector vs OpenSearch Serverless — side-by-side

**What you'll do:**
- Generate embeddings for 100 products using Amazon Titan Embed v2
- Store them in Aurora PostgreSQL (pgvector)
- Store the same vectors in OpenSearch Serverless
- Query both with the same natural language search
- Compare results and understand why they differ

**Prerequisites:**
- `make deploy-base` and `make deploy-web` completed
- `make seed-data` completed (products in Aurora and OpenSearch)
- Amazon Bedrock model access enabled for Titan Embeddings V2

**Estimated cost:** ~£0.01 (100 embedding calls to Titan Embed v2)

## Setup

In [10]:
import boto3
import json
import time
from botocore.exceptions import ClientError

# Configuration
AWS_REGION = "eu-west-1"       # Change if you deployed to a different region
AWS_PROFILE = "ridge-course-dev"  # Your AWS CLI profile name
STACK_NAME = "meridian-base"   # CloudFormation stack name

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
bedrock = session.client("bedrock-runtime")
rds_data = session.client("rds-data")

# Auto-discover all ARNs from CloudFormation stack outputs
cfn = session.client("cloudformation")
outputs = cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]
stack_out = {o["OutputKey"]: o["OutputValue"] for o in outputs}

CLUSTER_ARN = stack_out["AuroraClusterArn"]
SECRET_ARN = stack_out["AuroraSecretArn"]
OPENSEARCH_ENDPOINT = stack_out.get("OpenSearchCollectionEndpoint", "")
DATABASE = "meridian"

print(f"Region:     {AWS_REGION}")
print(f"Aurora:     {CLUSTER_ARN.split(':cluster:')[1]}")
print(f"Secret:     ...{SECRET_ARN[-12:]}")
print(f"OpenSearch: {OPENSEARCH_ENDPOINT[:60]}" if OPENSEARCH_ENDPOINT else "OpenSearch: NOT FOUND")


Region:     eu-west-1
Aurora:     meridian-base-auroracluster-ybtfvjqlngjr
Secret:     ...a/dev-uOY21x
OpenSearch: https://ylbjh3ualyui79xwz3qd.aoss.eu-west-1.on.aws


## Step 1: Select sample products from Aurora

We'll pick 100 products with good descriptions across different categories.

In [11]:
def execute_sql(sql, params=None):
    """Execute SQL via RDS Data API with auto-pause retry."""
    kwargs = {
        "resourceArn": CLUSTER_ARN,
        "secretArn": SECRET_ARN,
        "database": DATABASE,
        "sql": sql,
        "includeResultMetadata": True,
    }
    if params:
        kwargs["parameters"] = params
    for attempt in range(4):
        try:
            resp = rds_data.execute_statement(**kwargs)
            return resp
        except ClientError as exc:
            if "DatabaseResumingException" in str(exc) and attempt < 3:
                print(f"  Aurora resuming... waiting {3*(attempt+1)}s")
                time.sleep(3 * (attempt + 1))
                continue
            raise

# Fetch 100 products with descriptions, spread across categories
resp = execute_sql("""
    SELECT sku, name, l1, brand, short_description
    FROM products
    WHERE short_description IS NOT NULL
      AND length(short_description) > 50
    ORDER BY random()
    LIMIT 100
""")

columns = [col["name"] for col in resp["columnMetadata"]]
products = []
for row in resp["records"]:
    product = {}
    for i, col in enumerate(columns):
        field = row[i]
        if "stringValue" in field:
            product[col] = field["stringValue"]
        elif "isNull" in field:
            product[col] = None
        else:
            product[col] = str(field)
    products.append(product)

print(f"Selected {len(products)} products:")
for p in products[:5]:
    print(f"  {p['sku']} — {p['name']} ({p['l1']})")
print(f"  ... and {len(products)-5} more")

Selected 100 products:
  MER-PLW-MER-0088 — Loch Lomond Contour Pillow (Sleeping)
  MER-HLM-NOR-0112 — Glasgow Glacier Helmet (Ski & Snowsports)
  MER-SSJ-RES-0066 — Aviemore Explorer Jacket (Clothing)
  MER-TBV-NOR-0071 — Highland Ridge Tarp (Tents & Shelters)
  MER-WBF-BSC-0086 — Pen Y Fan Hydration Bottle (Camping & Cooking)
  ... and 95 more


## Step 2: Generate embeddings with Titan Embed v2

Each product's short description becomes a 1024-dimensional vector.

In [12]:
def generate_embedding(text):
    """Call Titan Embed v2 to get a 1024-dim embedding."""
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({
            "inputText": text,
            "dimensions": 1024,
            "normalize": True
        })
    )
    result = json.loads(response["body"].read())
    return result["embedding"]

# Generate embeddings for all products
print("Generating embeddings...")
for i, p in enumerate(products):
    text = f"{p['name']}. {p['short_description']}"
    p["embedding"] = generate_embedding(text)
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{len(products)} done")

print(f"\nAll {len(products)} embeddings generated.")
print(f"Vector dimensions: {len(products[0]['embedding'])}")
print(f"Sample values: [{products[0]['embedding'][0]:.4f}, {products[0]['embedding'][1]:.4f}, {products[0]['embedding'][2]:.4f}, ...]")

Generating embeddings...
  5/100 done
  10/100 done
  15/100 done
  20/100 done
  25/100 done
  30/100 done
  35/100 done
  40/100 done
  45/100 done
  50/100 done
  55/100 done
  60/100 done
  65/100 done
  70/100 done
  75/100 done
  80/100 done
  85/100 done
  90/100 done
  95/100 done
  100/100 done

All 100 embeddings generated.
Vector dimensions: 1024
Sample values: [-0.0541, -0.0133, 0.0113, ...]


## Step 3: Store in Aurora pgvector

We update the existing `embedding` column on the products table.

In [13]:
# Ensure pgvector extension exists
execute_sql("CREATE EXTENSION IF NOT EXISTS vector")

# Store embeddings in the products table
stored = 0
for p in products:
    vec_str = "[" + ",".join(str(v) for v in p["embedding"]) + "]"
    execute_sql(
        "UPDATE products SET embedding = :vec::vector WHERE sku = :sku",
        params=[
            {"name": "vec", "value": {"stringValue": vec_str}},
            {"name": "sku", "value": {"stringValue": p["sku"]}},
        ]
    )
    stored += 1

print(f"Stored {stored} embeddings in Aurora pgvector")

Stored 100 embeddings in Aurora pgvector


## Step 4: Store in OpenSearch Serverless

We index the same vectors into the existing OpenSearch `products` index, adding a `description_embedding` field.

In [14]:
import hashlib
import urllib.request
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

def opensearch_request(method, path, body=None):
    """Make a SigV4-signed request to OpenSearch Serverless."""
    url = f"{OPENSEARCH_ENDPOINT}{path}"
    data = json.dumps(body).encode() if body else b""
    content_hash = hashlib.sha256(data).hexdigest()
    headers = {
        "Content-Type": "application/json",
        "x-amz-content-sha256": content_hash,
    }
    credentials = session.get_credentials().get_frozen_credentials()
    request = AWSRequest(method=method, url=url, data=data, headers=headers)
    SigV4Auth(credentials, "aoss", AWS_REGION).add_auth(request)
    req = urllib.request.Request(url=request.url, data=data, headers=dict(request.headers), method=method)
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())

# Update each product document with its embedding
indexed = 0
for p in products:
    doc = {"doc": {"description_embedding": p["embedding"]}}
    try:
        opensearch_request("POST", f"/products/_update/{p['sku']}", doc)
        indexed += 1
    except Exception as e:
        print(f"  Failed to update {p['sku']}: {e}")

print(f"Indexed {indexed} embeddings in OpenSearch")

Indexed 100 embeddings in OpenSearch


## Step 5: Query both stores with the same search

Let's search for "warm waterproof jacket for hiking in Scotland" and compare results.

In [15]:
QUERY = "warm waterproof jacket for hiking in Scotland"

# Generate the query embedding
query_embedding = generate_embedding(QUERY)
print(f"Query: '{QUERY}'")
print(f"Query vector: [{query_embedding[0]:.4f}, {query_embedding[1]:.4f}, ...] (1024 dims)")

Query: 'warm waterproof jacket for hiking in Scotland'
Query vector: [-0.0528, -0.0381, ...] (1024 dims)


### pgvector query (pure cosine similarity)

In [16]:
# Query pgvector — find nearest neighbours by cosine distance
vec_str = "[" + ",".join(str(v) for v in query_embedding) + "]"
pg_start = time.time()
resp = execute_sql(
    """SELECT sku, name, l1, brand,
              1 - (embedding <=> :query::vector) as similarity
       FROM products
       WHERE embedding IS NOT NULL
       ORDER BY embedding <=> :query::vector
       LIMIT 5""",
    params=[{"name": "query", "value": {"stringValue": vec_str}}]
)
pg_time = (time.time() - pg_start) * 1000

columns = [col["name"] for col in resp["columnMetadata"]]
pg_results = []
for row in resp["records"]:
    r = {}
    for i, col in enumerate(columns):
        field = row[i]
        if "stringValue" in field:
            r[col] = field["stringValue"]
        elif "doubleValue" in field:
            r[col] = field["doubleValue"]
        elif "isNull" in field:
            r[col] = None
    pg_results.append(r)

print(f"pgvector results ({pg_time:.0f}ms):")
print(f"{'Rank':<5} {'SKU':<20} {'Name':<40} {'Similarity':<10}")
print("-" * 75)
for i, r in enumerate(pg_results, 1):
    sim = f"{r.get('similarity', 0):.4f}" if r.get('similarity') else "N/A"
    print(f"{i:<5} {r['sku']:<20} {r['name'][:38]:<40} {sim}")

pgvector results (358ms):
Rank  SKU                  Name                                     Similarity
---------------------------------------------------------------------------
1     MER-JKT-STO-0129     Cairngorm 500 Jacket                     0.5189
2     MER-JKT-MER-0011     Snowdonia Windproof Jacket               0.4699
3     MER-WPJ-RES-0063     Brecon Beacon Trail Jacket               0.4686
4     MER-JKT-STO-0120     Edinburgh Lightweight Jacket             0.4615
5     MER-WPJ-NOR-0060     Isle Of Skye Explorer Jacket             0.4572


### OpenSearch query (hybrid: BM25 + vector)

In [17]:
# Query OpenSearch — knn vector similarity on the demo index
os_body = {
    "query": {
        "knn": {
            "embedding_vector": {
                "vector": query_embedding,
                "k": 5
            }
        }
    },
    "size": 5,
    "_source": ["sku", "name", "l1", "brand"]
}

os_start = time.time()
result = opensearch_request("POST", f"/{DEMO_INDEX}/_search", os_body)
os_time = (time.time() - os_start) * 1000
hits = result.get("hits", {}).get("hits", [])

os_results = [{**h["_source"], "score": h["_score"]} for h in hits]

print(f"\nOpenSearch knn results ({os_time:.0f}ms):")
print(f"{'Rank':<5} {'SKU':<20} {'Name':<40} {'Score':<10}")
print("-" * 75)
for i, r in enumerate(os_results, 1):
    print(f"{i:<5} {r['sku']:<20} {r['name'][:38]:<40} {r.get('score', 0):.4f}")


  (Note: knn not available, using BM25 only)

OpenSearch results (17486ms):
Rank  SKU                  Name                                     Score     
---------------------------------------------------------------------------
1     MER-SKJ-PDC-0030     Stoke On Trent Warm Jacket               12.0729
2     MER-JKT-RES-0052     Anglesey Waterproof Jacket               12.0246
3     MER-INJ-CAI-0044     Tay Valley Warm Jacket                   11.3783
4     MER-FLC-RES-0036     Isle Of Skye Warm Jacket                 11.3783
5     MER-JKT-STO-0094     Pembroke Waterproof Jacket               11.1134


## Step 6: Compare results side-by-side

In [ ]:
print(f"\n{'='*80}")
print(f"COMPARISON: '{QUERY}'")
print(f"{'='*80}")
print(f"\n{'pgvector (pure vector, cosine similarity)':<45} {'OpenSearch (hybrid BM25 + vector)'}")
print(f"{'Latency: ' + f'{pg_time:.0f}ms':<45} {'Latency: ' + f'{os_time:.0f}ms'}")
print(f"{'-'*45} {'-'*35}")

max_rows = max(len(pg_results), len(os_results))
for i in range(max_rows):
    pg_name = pg_results[i]["name"][:40] if i < len(pg_results) else ""
    os_name = os_results[i]["name"][:33] if i < len(os_results) else ""
    print(f"{i+1}. {pg_name:<43} {i+1}. {os_name}")

print(f"\n{'='*80}")
print("Key insight: same vectors, different ranking.")
print("pgvector ranks purely by vector distance (meaning similarity).")
print("OpenSearch combines keyword match score with vector similarity.")
print(f"{'='*80}")

## What you've built

- Generated real embeddings using Amazon Titan Embed v2
- Stored vectors in two different stores (pgvector and OpenSearch)
- Queried both with the same natural language input
- Observed how scoring algorithms produce different rankings

**Key takeaway:** The vector store you choose affects not just cost and operations, but *which results your users see*. OpenSearch's hybrid approach finds products matching both keywords and meaning. pgvector's pure cosine similarity finds the semantically closest matches regardless of exact term overlap.

## Optional extensions

1. Try more queries: "lightweight tent for summer festivals", "birthday gift for a climber under £50"
2. Increase the sample to 100 or 500 products and observe latency changes
3. Compare results with 256-dim vs 1024-dim embeddings (change the `dimensions` parameter)